In [67]:
import numpy as np
import pandas as pd
import gzip
import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer #FOR TF-IDF (FINDING IMPORTANT WORDS)
from sklearn.neighbors import BallTree #FOR SPATIAL ANALYSIS (NEIGHBOR DENSITY)

In [68]:

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=1000000):
    g = gzip.open('data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

state_reviews = pd.DataFrame(parse_first_n("review-Virginia.json.gz"))
state_meta = pd.DataFrame(parse("data/meta-Virginia.json.gz"))

## CLASSIFYING AREA PART

In [69]:
state_merged_reviews = state_meta.merge(state_reviews, on='gmap_id')
coords = state_merged_reviews[['latitude', 'longitude']].dropna().values
coords_rad = np.radians(coords)
df_valid = state_merged_reviews[['latitude','longitude']].dropna().copy()
coords_rad = np.radians(df_valid[['latitude','longitude']].values)

tree = BallTree(coords_rad, metric='haversine')
radius_miles = 5
radius = radius_miles / 3959
counts = tree.query_radius(coords_rad, r=radius, count_only=True)

df_valid['local_density'] = counts - 1
state_merged_reviews.loc[df_valid.index, 'local_density'] = df_valid['local_density']
state_merged_reviews['density_bin'] = pd.qcut(
    state_merged_reviews['local_density'],
    q=3,
    labels=['Rural','Suburban','Urban']
)
state_merged_reviews = state_merged_reviews[~state_merged_reviews.get('text').isna()]

In [70]:
## Maps of Important Words
restaurant_dict = {}
nonrestaurant_dict = {}

### URBAN

In [71]:
state_urban_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Urban']
urban_state_reviews_good = state_urban_reviews[(state_urban_reviews.get('rating') == 4) | (state_urban_reviews.get('rating') == 5)]
urban_state_reviews_bad = state_urban_reviews[(state_urban_reviews.get('rating') == 1) | (state_urban_reviews.get('rating') == 2)]


#### Restaurant Good

In [72]:
state_urban_restaurant_reviews_good = urban_state_reviews_good[urban_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_urban_state_restaurant_reviews_good = state_urban_restaurant_reviews_good.get('text').tolist()

rest_urban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_urban_result_good = rest_urban_tfidf_good.fit_transform(list_of_urban_state_restaurant_reviews_good)

rest_urban_feature_names_good = rest_urban_tfidf_good.get_feature_names_out()
rest_mean_urban_tfidf_good = np.asarray(rest_urban_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_urban = rest_mean_urban_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_urban = [(rest_urban_feature_names_good[i], rest_mean_urban_tfidf_good[i]) for i in rest_top_indices_good_urban]

rows = []

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Restaurant",
        "review_group": "Good"
    })

#### Non-Restaurant Good

In [73]:
state_urban_nonrestaurant_reviews_good = urban_state_reviews_good[
    urban_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_urban_state_nonrestaurant_reviews_good = state_urban_nonrestaurant_reviews_good.get('text').tolist()

nonrest_urban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_urban_result_good = nonrest_urban_tfidf_good.fit_transform(list_of_urban_state_nonrestaurant_reviews_good)

nonrest_urban_feature_names_good = nonrest_urban_tfidf_good.get_feature_names_out()
nonrest_mean_urban_tfidf_good = np.asarray(nonrest_urban_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_urban = nonrest_mean_urban_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_urban = [(nonrest_urban_feature_names_good[i], nonrest_mean_urban_tfidf_good[i]) for i in nonrest_top_indices_good_urban]

for word, score in nonrest_top_words_good_urban:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in nonrest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })


highly recommend 0.01179385702009861
customer service 0.009642307789989292
great service 0.008179345191194596
great experience 0.005683160081723963
great job 0.005420149546944189
great place 0.004589456354889817
great customer 0.004351291798628728
excellent service 0.004345426228720908
friendly staff 0.00383451014081093
definitely recommend 0.003742974237871348
did great 0.0033048292461566892
staff friendly 0.0032124593959952687
translated google 0.0031781780282215772
highly recommended 0.0029855754161418023
great work 0.002682861366667757
friendly helpful 0.0026486817988271485
good service 0.0024542192602738616
great staff 0.0024192117575710026
make sure 0.0023761188214948136
friendly professional 0.0023021565061859283


#### Restaurant Bad

In [74]:
state_urban_restaurant_reviews_bad = urban_state_reviews_bad[urban_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_urban_state_restaurant_reviews_bad = state_urban_restaurant_reviews_bad.get('text').tolist()

rest_urban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_urban_result_bad = rest_urban_tfidf_bad.fit_transform(list_of_urban_state_restaurant_reviews_bad)

rest_urban_feature_names_bad = rest_urban_tfidf_bad.get_feature_names_out()
rest_mean_urban_tfidf_bad = np.asarray(rest_urban_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_urban = rest_mean_urban_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_urban = [(rest_urban_feature_names_bad[i], rest_mean_urban_tfidf_bad[i]) for i in rest_top_indices_bad_urban]

for word, score in rest_top_words_bad_urban:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

customer service 0.028091589521840203
translated google 0.025743597321427468
food good 0.012612837238722797
bad service 0.008632777314617118
tasted like 0.008523860023761222
30 minutes 0.0074177508910960544
don know 0.006738054712287576
waste money 0.006446285280803511
slow service 0.006270717428383505
worst experience 0.006015847967397434
good food 0.005932856632350743
horrible service 0.005823172944252084
service good 0.005727082590728704
20 minutes 0.0056268170028857594
terrible service 0.005499310089585746
15 minutes 0.005395249536423777
food cold 0.005251230411973177
poor service 0.005210171100808121
looks like 0.005173171208706007
bad experience 0.0050273463532420634


#### Non-Restaurant Bad

In [75]:
state_urban_nonrestaurant_reviews_bad = urban_state_reviews_bad[
    urban_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_urban_state_nonrestaurant_reviews_bad = state_urban_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_urban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_urban_result_bad = nonrest_urban_tfidf_bad.fit_transform(list_of_urban_state_nonrestaurant_reviews_bad)

nonrest_urban_feature_names_bad = nonrest_urban_tfidf_bad.get_feature_names_out()
nonrest_mean_urban_tfidf_bad = np.asarray(nonrest_urban_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_urban = nonrest_mean_urban_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_urban = [(nonrest_urban_feature_names_bad[i], nonrest_mean_urban_tfidf_bad[i]) for i in nonrest_top_indices_bad_urban]

for word, score in nonrest_top_words_bad_urban:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.01782081840400581
waste time 0.004750099625503628
stay away 0.004611318514185575
don know 0.004017549184119546
translated google 0.0038851145206359777
poor customer 0.0033439343329512063
worst customer 0.0032202358349683034
answer phone 0.00282231874480966
horrible customer 0.002774547024395514
bad service 0.0027242876392261023
worst experience 0.0027226628779860296
bad customer 0.002659069599516615
terrible customer 0.0026329117329005136
make sure 0.002631145879567206
don care 0.002517827521876457
30 minutes 0.0024818016103359247
bad experience 0.0024797019911850563
extremely rude 0.002476686391413378
recommend place 0.0024462581354840916
horrible experience 0.0023704964955426114


### SUBURBAN

In [76]:
state_suburban_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Suburban']
suburban_state_reviews_good = state_suburban_reviews[(state_suburban_reviews.get('rating') == 4) | (state_suburban_reviews.get('rating') == 5)]
suburban_state_reviews_bad = state_suburban_reviews[(state_suburban_reviews.get('rating') == 1) | (state_suburban_reviews.get('rating') == 2)]


#### Restaurant Good

In [77]:
state_suburban_restaurant_reviews_good = suburban_state_reviews_good[suburban_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_suburban_state_restaurant_reviews_good = state_suburban_restaurant_reviews_good.get('text').tolist()

rest_suburban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_suburban_result_good = rest_suburban_tfidf_good.fit_transform(list_of_suburban_state_restaurant_reviews_good)

rest_suburban_feature_names_good = rest_suburban_tfidf_good.get_feature_names_out()
rest_mean_suburban_tfidf_good = np.asarray(rest_suburban_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_suburban = rest_mean_suburban_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_suburban = [(rest_suburban_feature_names_good[i], rest_mean_suburban_tfidf_good[i]) for i in rest_top_indices_good_suburban]

for word, score in rest_top_words_good_suburban:
    restaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Restaurant",
        "review_group": "Good"
    })


great food 0.027929136510330452
good food 0.023721516452328298
food great 0.01784203700122195
great service 0.015214726240419998
food good 0.014600667858102555
customer service 0.013249857097393913
translated google 0.010313592036489136
food delicious 0.009701577004511195
highly recommend 0.009505716639234147
food service 0.009165366996999415
love place 0.009097590106882504
really good 0.008506406143268221
friendly staff 0.00849442237392738
great place 0.008230824090674618
delicious food 0.007675782303058282
good service 0.007612109384642664
amazing food 0.0074508806343583835
food amazing 0.007386250989439322
service great 0.007373112555675054
excellent food 0.006728958653351576


#### Non-Restaurant Good

In [78]:
state_suburban_nonrestaurant_reviews_good = suburban_state_reviews_good[
    suburban_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_suburban_state_nonrestaurant_reviews_good = state_suburban_nonrestaurant_reviews_good.get('text').tolist()

nonrest_suburban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_suburban_result_good = nonrest_suburban_tfidf_good.fit_transform(list_of_suburban_state_nonrestaurant_reviews_good)

nonrest_suburban_feature_names_good = nonrest_suburban_tfidf_good.get_feature_names_out()
nonrest_mean_suburban_tfidf_good = np.asarray(nonrest_suburban_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_suburban = nonrest_mean_suburban_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_suburban = [(nonrest_suburban_feature_names_good[i], nonrest_mean_suburban_tfidf_good[i]) for i in nonrest_top_indices_good_suburban]

for word, score in nonrest_top_words_good_suburban:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })

highly recommend 0.011126395652371453
customer service 0.010633473570082872
great service 0.009278836408333414
great experience 0.0059934154553763445
great place 0.00585350297029526
great job 0.005308443861209975
great customer 0.004951860649334629
friendly staff 0.004704539613012861
excellent service 0.004548261893857075
staff friendly 0.003887038536866639
definitely recommend 0.0037616136650680527
friendly helpful 0.0032343887154870155
good service 0.0031930701683790265
did great 0.003170375119398027
great work 0.0028811676887165083
great staff 0.002865569799359314
translated google 0.002842110932247441
highly recommended 0.0027550961850872848
love place 0.0027313426209320067
service great 0.0026838457450051928


#### Restaurant Bad

In [79]:
state_suburban_restaurant_reviews_bad = suburban_state_reviews_bad[suburban_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_suburban_state_restaurant_reviews_bad = state_suburban_restaurant_reviews_bad.get('text').tolist()

rest_suburban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_suburban_result_bad = rest_suburban_tfidf_bad.fit_transform(list_of_suburban_state_restaurant_reviews_bad)

rest_suburban_feature_names_bad = rest_suburban_tfidf_bad.get_feature_names_out()
rest_mean_suburban_tfidf_bad = np.asarray(rest_suburban_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_suburban = rest_mean_suburban_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_suburban = [(rest_suburban_feature_names_bad[i], rest_mean_suburban_tfidf_bad[i]) for i in rest_top_indices_bad_suburban]

for word, score in rest_top_words_bad_suburban:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

customer service 0.027538571727092422
translated google 0.01442994172969981
food good 0.012065653002542077
fried rice 0.009669293204170724
tasted like 0.009029278721526299
dunkin donuts 0.006466225983415248
food cold 0.006386693371396467
don know 0.006360495680526071
bad service 0.006321396525419297
waste money 0.006152108195980866
20 minutes 0.005513161690672014
30 minutes 0.005489655508061001
10 minutes 0.005445540964335095
terrible service 0.005404161817029961
quality food 0.005305768919495813
poor customer 0.005185822830219801
slow service 0.005153106120495083
chinese food 0.005036666290896218
poor service 0.005034579082538657
good food 0.004918472450078865


#### Non-Restaurant Bad

In [80]:
state_suburban_nonrestaurant_reviews_bad = suburban_state_reviews_bad[
    suburban_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_suburban_state_nonrestaurant_reviews_bad = state_suburban_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_suburban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_suburban_result_bad = nonrest_suburban_tfidf_bad.fit_transform(list_of_suburban_state_nonrestaurant_reviews_bad)

nonrest_suburban_feature_names_bad = nonrest_suburban_tfidf_bad.get_feature_names_out()
nonrest_mean_suburban_tfidf_bad = np.asarray(nonrest_suburban_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_suburban = nonrest_mean_suburban_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_suburban = [(nonrest_suburban_feature_names_bad[i], nonrest_mean_suburban_tfidf_bad[i]) for i in nonrest_top_indices_bad_suburban]

for word, score in nonrest_top_words_bad_suburban:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.015724500483819664
waste time 0.004643931268288487
translated google 0.003762001651420591
don know 0.003671035953342865
stay away 0.0036186431783601066
poor customer 0.0029929527018126914
answer phone 0.0028017476536409975
make sure 0.00276210934646482
worst experience 0.002709143123125985
worst customer 0.002623420204652988
recommend place 0.0024960518583590414
extremely rude 0.0024905275655042625
don care 0.002416139568581599
worst place 0.0023079333482852364
don waste 0.0022942653543227593
30 minutes 0.0022417048407821676
terrible customer 0.002187843939847129
zero stars 0.002120936646550729
multiple times 0.002119565369386413
horrible customer 0.00211355002518093


### RURAL

In [81]:
state_rural_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Rural']
rural_state_reviews_good = state_rural_reviews[(state_rural_reviews.get('rating') == 4) | (state_rural_reviews.get('rating') == 5)]
rural_state_reviews_bad = state_rural_reviews[(state_rural_reviews.get('rating') == 1) | (state_rural_reviews.get('rating') == 2)]


#### Restaurant Good

In [82]:
state_rural_restaurant_reviews_good = rural_state_reviews_good[rural_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_rural_state_restaurant_reviews_good = state_rural_restaurant_reviews_good.get('text').tolist()

rest_rural_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_rural_result_good = rest_rural_tfidf_good.fit_transform(list_of_rural_state_restaurant_reviews_good)

rest_rural_feature_names_good = rest_rural_tfidf_good.get_feature_names_out()
rest_mean_rural_tfidf_good = np.asarray(rest_rural_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_rural = rest_mean_rural_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_rural = [(rest_rural_feature_names_good[i], rest_mean_rural_tfidf_good[i]) for i in rest_top_indices_good_rural]

for word, score in rest_top_words_good_rural:
    restaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Restaurant",
        "review_group": "Good"
    })


great food 0.038771046520869365
good food 0.02907757583699967
food great 0.02056307860817123
great service 0.01953666793371195
food good 0.015758272075700304
friendly staff 0.012671910921012554
food service 0.012514322273674586
great place 0.012004064268147215
service great 0.010892010773027882
customer service 0.010119117210877663
food delicious 0.009391645617813483
highly recommend 0.009127639012966714
good service 0.00891618704039288
excellent food 0.008424892277848985
really good 0.008192897590665077
love place 0.007707635113855481
delicious food 0.007152996118702241
friendly service 0.007134957535752617
staff friendly 0.0070260563177288226
food friendly 0.006998874509185094


#### Non-Restaurant Good

In [83]:
state_rural_nonrestaurant_reviews_good = rural_state_reviews_good[
    rural_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_rural_state_nonrestaurant_reviews_good = state_rural_nonrestaurant_reviews_good.get('text').tolist()

nonrest_rural_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_rural_result_good = nonrest_rural_tfidf_good.fit_transform(list_of_rural_state_nonrestaurant_reviews_good)

nonrest_rural_feature_names_good = nonrest_rural_tfidf_good.get_feature_names_out()
nonrest_mean_rural_tfidf_good = np.asarray(nonrest_rural_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_rural = nonrest_mean_rural_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_rural = [(nonrest_rural_feature_names_good[i], nonrest_mean_rural_tfidf_good[i]) for i in nonrest_top_indices_good_rural]

for word, score in nonrest_top_words_good_rural:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })

great service 0.010679123133392112
highly recommend 0.01053320706256853
great place 0.009698920873948715
customer service 0.00915900109939031
great job 0.00608441514860228
friendly staff 0.0054699732176551686
great experience 0.0047783071245179306
great people 0.004771423164768641
excellent service 0.004480673907828526
great customer 0.00440145567829243
friendly helpful 0.003780658363478126
nice people 0.003419581968314517
love place 0.0033357892435534674
did great 0.0032974533156740096
great work 0.0032373062618692573
staff friendly 0.0031826315874064395
good service 0.0031301788568085112
definitely recommend 0.0030846706800585436
great staff 0.0027666394473227192
service great 0.0027270632128606333


#### Restaurant Bad

In [84]:
state_rural_restaurant_reviews_bad = rural_state_reviews_bad[rural_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_rural_state_restaurant_reviews_bad = state_rural_restaurant_reviews_bad.get('text').tolist()

rest_rural_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_rural_result_bad = rest_rural_tfidf_bad.fit_transform(list_of_rural_state_restaurant_reviews_bad)

rest_rural_feature_names_bad = rest_rural_tfidf_bad.get_feature_names_out()
rest_mean_rural_tfidf_bad = np.asarray(rest_rural_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_rural = rest_mean_rural_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_rural = [(rest_rural_feature_names_bad[i], rest_mean_rural_tfidf_bad[i]) for i in rest_top_indices_bad_rural]

for word, score in rest_top_words_bad_rural:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

customer service 0.019915623684690034
food good 0.012552442378900555
translated google 0.009974575272585603
tasted like 0.009440110615891761
20 minutes 0.009426406391158876
slow service 0.006885078282993472
waste time 0.006253758964830341
don know 0.006025206107525708
30 minutes 0.005993360784324774
poor service 0.005976098575817973
fried rice 0.005925154571338892
good food 0.005904151207884236
15 minutes 0.0058865179208850565
food terrible 0.005449681839169398
food ok 0.005276743407165511
ordered food 0.00472001408410953
parking lot 0.004634261073311871
chinese food 0.004618545914132244
service terrible 0.0045246814626809856
terrible service 0.0045087202160878906


#### Non-Restaurant Bad

In [85]:
state_rural_nonrestaurant_reviews_bad = rural_state_reviews_bad[
    rural_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_rural_state_nonrestaurant_reviews_bad = state_rural_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_rural_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_rural_result_bad = nonrest_rural_tfidf_bad.fit_transform(list_of_rural_state_nonrestaurant_reviews_bad)

nonrest_rural_feature_names_bad = nonrest_rural_tfidf_bad.get_feature_names_out()
nonrest_mean_rural_tfidf_bad = np.asarray(nonrest_rural_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_rural = nonrest_mean_rural_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_rural = [(nonrest_rural_feature_names_bad[i], nonrest_mean_rural_tfidf_bad[i]) for i in nonrest_top_indices_bad_rural]

for word, score in nonrest_top_words_bad_rural:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.013370963722574273
waste time 0.004444375478815488
don know 0.004326344143925997
stay away 0.003826723668626403
translated google 0.003303472341542124
poor customer 0.003000502216985433
answer phone 0.002920601674776787
make sure 0.002788388152673256
zero stars 0.002373186907378334
parking lot 0.002276052349674335
post office 0.0022582786361489767
poor service 0.0022508557197663476
worst place 0.0022495791655218972
don waste 0.0022002158667569567
don care 0.0021880293758800382
phone calls 0.002176212004853958
extremely rude 0.0021751416545903093
30 minutes 0.0021662544536922
let know 0.0021590991686834186
year old 0.0020920020208807285


In [86]:
bigram_df = pd.DataFrame(rows)

In [87]:
bigram_df

,bigram,score,area_type,business_type,review_group
0,great food,0.025598,Urban,Restaurant,Good
1,good food,0.018373,Urban,Restaurant,Good
2,translated google,0.016456,Urban,Restaurant,Good
3,food great,0.013431,Urban,Restaurant,Good
4,customer service,0.013278,Urban,Restaurant,Good
...,...,...,...,...,...
235,food amazing,0.007455,Rural,Non-Restaurant,Bad
236,love place,0.007197,Rural,Non-Restaurant,Bad
237,good service,0.007113,Rural,Non-Restaurant,Bad
238,amazing food,0.007010,Rural,Non-Restaurant,Bad


In [88]:
bigram_df.to_csv("virginia_insights.csv", index=False)